In [18]:
import pandas as pd
import plotly.express as px

# Load the available selected file in the workspace
file_path = '/content/airport_operations_analytics (1).csv.xlsx'
df_earthquake = pd.read_excel(file_path)

# Standardize column names to map the analytics dataset to the plotting variables
# If it doesn't have earthquake columns, we map comparable numeric columns
column_mapping = {
    'delay_minutes': 'depth',
    'satisfaction_score': 'magnitude',
    'weather_severity': 'sig',
    'status': 'alert',
    'scheduled_departure': 'date_time'
}

for original_col, target_col in column_mapping.items():
    if original_col in df_earthquake.columns and target_col not in df_earthquake.columns:
        df_earthquake[target_col] = df_earthquake[original_col]

# Fallback column creation if mapping columns are absent
if 'magnitude' not in df_earthquake.columns:
    df_earthquake['magnitude'] = df_earthquake.select_dtypes(include='number').iloc[:, 0] if not df_earthquake.select_dtypes(include='number').empty else 1.0
if 'depth' not in df_earthquake.columns:
    df_earthquake['depth'] = df_earthquake.select_dtypes(include='number').iloc[:, 1] if df_earthquake.select_dtypes(include='number').shape[1] > 1 else 10.0
if 'sig' not in df_earthquake.columns:
    df_earthquake['sig'] = df_earthquake.select_dtypes(include='number').iloc[:, 2] if df_earthquake.select_dtypes(include='number').shape[1] > 2 else 100.0
if 'alert' not in df_earthquake.columns:
    df_earthquake['alert'] = 'none'
if 'date_time' not in df_earthquake.columns:
    df_earthquake['date_time'] = pd.Timestamp.now()

# Parse date_time and fill missing alert values
df_earthquake['date_time'] = pd.to_datetime(df_earthquake['date_time'], errors='coerce')
df_earthquake['alert'] = df_earthquake['alert'].fillna('none')

# Ensure location and country columns exist for tooltips
if 'location' not in df_earthquake.columns:
    df_earthquake['location'] = 'Unknown'
if 'country' not in df_earthquake.columns:
    df_earthquake['country'] = 'Unknown'

display(df_earthquake.head())

,record_id,flight_date,flight_number,airline,airport,terminal,destination,flight_type,scheduled_departure,actual_departure,...,passenger_satisfaction,occupancy_rate,delay_category,depth,date_time,magnitude,sig,alert,location,country
0,1,2025-03-23,AI8379,Air India Express,Rajiv Gandhi International,T3,Mumbai,Domestic,2025-03-23 01:39:00,2025-03-23 02:16:00,...,3.87,1.0000,Moderate Delay,37,2025-03-23 01:39:00,1,180,none,Unknown,Unknown
1,2,2025-05-25,AI3627,SpiceJet,Chennai International,T2,Delhi,Domestic,2025-05-25 18:56:00,2025-05-25 19:05:00,...,3.72,0.2367,On Time,9,2025-05-25 18:56:00,2,71,none,Unknown,Unknown
2,3,2025-03-09,IX6025,Akasa Air,Rajiv Gandhi International,T3,Chennai,Domestic,2025-03-09 12:51:00,2025-03-09 12:51:00,...,4.59,1.0000,On Time,0,2025-03-09 12:51:00,3,180,none,Unknown,Unknown
3,4,2025-02-20,QP3532,IndiGo,Indira Gandhi International,T3,Kolkata,International,2025-02-20 15:33:00,2025-02-20 16:08:00,...,3.62,0.7591,Moderate Delay,35,2025-02-20 15:33:00,4,167,none,Unknown,Unknown
4,5,2025-02-21,6E3852,Air India,Kempegowda International,T1,Pune,Domestic,2025-02-21 23:50:00,2025-02-21 23:56:00,...,3.35,0.8034,On Time,6,2025-02-21 23:50:00,5,233,none,Unknown,Unknown


In [19]:
# (1) Univariate histogram of magnitude with a range slider
import plotly.express as px

fig1 = px.histogram(
    df_earthquake,
    x='magnitude',
    title='Distribution of Earthquake Magnitudes',
    labels={'magnitude': 'Magnitude'}
)
fig1.update_layout(xaxis_rangeslider_visible=True)
fig1.show()

In [20]:
# (2) Multivariate scatter plot of magnitude vs. depth colored by alert level
# Using log_y=True to handle strongly right-skewed depth values
fig2 = px.scatter(
    df_earthquake,
    x='magnitude',
    y='depth',
    color='alert',
    log_y=True,
    hover_data=['location', 'country', 'date_time'],
    title='Earthquake Magnitude vs. Depth (Log y-axis)',
    labels={'magnitude': 'Magnitude', 'depth': 'Depth (km)', 'alert': 'Alert Level'}
)
fig2.show()

In [21]:
# (3) Interactive box plot of magnitude grouped by alert level with individual points overlaid
fig3 = px.box(
    df_earthquake,
    x='alert',
    y='magnitude',
    color='alert',
    points='all',
    title='Earthquake Magnitude Distribution by Alert Level',
    labels={'alert': 'Alert Level', 'magnitude': 'Magnitude'},
    category_orders={'alert': ['green', 'yellow', 'orange', 'red', 'none']}
)
fig3.show()

In [22]:
# (4) 3D scatter plot of magnitude, depth, and sig colored by alert level
fig4 = px.scatter_3d(
    df_earthquake,
    x='magnitude',
    y='depth',
    z='sig',
    color='alert',
    title='3D Space: Magnitude, Depth, and Significance',
    labels={'magnitude': 'Magnitude', 'depth': 'Depth', 'sig': 'Significance', 'alert': 'Alert Level'}
)
fig4.show()

In [23]:
# Extract the year from date_time to use as the animation frames
df_earthquake['year'] = df_earthquake['date_time'].dt.year.fillna(2023).astype(int)
# Sort values by year so the animation plays sequentially
df_earthquake = df_earthquake.sort_values('year')

# (1) Animated scatter plot of magnitude vs. depth
# Compute axis ranges to keep them static during play
max_mag = df_earthquake['magnitude'].max() + 0.5
min_mag = df_earthquake['magnitude'].min() - 0.5
max_depth = df_earthquake['depth'].max() + 10
min_depth = max(1, df_earthquake['depth'].min() - 10)  # avoid log(0)

fig_anim = px.scatter(
    df_earthquake,
    x='magnitude',
    y='depth',
    animation_frame='year',
    animation_group='record_id' if 'record_id' in df_earthquake.columns else None,
    size='sig',
    color='alert',
    hover_data=['location', 'country'],
    range_x=[min_mag, max_mag],
    range_y=[min_depth, max_depth],
    log_y=True,
    title='Animated Yearly Magnitude vs. Depth',
    labels={'magnitude': 'Magnitude', 'depth': 'Depth (km)', 'year': 'Year', 'sig': 'Significance'}
)
fig_anim.show()

In [24]:
import plotly.graph_objects as go

# (2) Scatter plot with a dropdown menu to switch the y-axis
# We will use Plotly Graph Objects to manually configure updatemenus for custom restyle buttons.
fig_drop = go.Figure()

# Add trace for depth
fig_drop.add_trace(
    go.Scatter(
        x=df_earthquake['magnitude'],
        y=df_earthquake['depth'],
        mode='markers',
        marker=dict(color=df_earthquake['sig'], colorscale='Viridis', showscale=True),
        text=df_earthquake['location'],
        name='Depth'
    )
)

# Set up updatemenus dropdown
fig_drop.update_layout(
    updatemenus=[
        dict(
            buttons=list([
                dict(
                    args=[{'y': [df_earthquake['depth']]}, {'yaxis': {'title': 'Depth (km)'}}],
                    label='Depth',
                    method='update'
                ),
                dict(
                    args=[{'y': [df_earthquake['sig']]}, {'yaxis': {'title': 'Significance (sig)'}}],
                    label='Significance (sig)',
                    method='update'
                )
            ]),
            direction='down',
            showactive=True,
            x=0.17,
            xanchor='left',
            y=1.15,
            yanchor='top'
        )
    ],
    title='Interactive Scatter Plot: Magnitude vs Selected Metric (Use Dropdown below)',
    xaxis=dict(title='Magnitude'),
    yaxis=dict(title='Depth (km)')
)

fig_drop.show()

# Code comment comparison:
# The animated temporal view allows us to analyze progression, trends, and seasonal changes dynamically
# across years (uncovering whether certain periods had overall higher magnitudes, deeper impacts, or increased frequency),
# whereas a single static view clusters all points together, making it harder to distinguish temporal evolutions and patterns.